In [4]:
import os
import pandas as pd

# Base directory
base_dir = "/home/gauss/Documents/Project 2: Combining Molecular and Imagining Data for MOA Prediction/smile_image_global/Image-based Model/results/"

all_dfs = []

# Collect all subfolders first (to build mapping) within the cell line folder
subfolders = [f for f in os.listdir(base_dir) 
              if os.path.isdir(os.path.join(base_dir, f)) and f.startswith("HEPG2_new")]

# Create mapping: U2OS_... -> Model 1, Model 2, ...
model_map = {name: f"M{i+1}" for i, name in enumerate(sorted(subfolders))}

# Read CSVs and merge
for subfolder in subfolders:
    csv_path = os.path.join(base_dir, subfolder, "fold_metrics.csv")
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df["Model"] = subfolder
        df["Easy_Model_Name"] = model_map[subfolder]
        all_dfs.append(df)

merged_df = pd.concat(all_dfs, ignore_index=True)
merged_df


,fold,acc,precision,recall,f1,Model,Easy_Model_Name
0,1,0.648889,0.508255,0.573046,0.527482,HEPG2_new_mean_standard_stratified_merge_all_k...,M26
1,2,0.640000,0.516573,0.556441,0.525270,HEPG2_new_mean_standard_stratified_merge_all_k...,M26
2,3,0.688889,0.506964,0.541331,0.517940,HEPG2_new_mean_standard_stratified_merge_all_k...,M26
3,4,0.608889,0.441314,0.529212,0.473316,HEPG2_new_mean_standard_stratified_merge_all_k...,M26
4,5,0.648889,0.481174,0.569444,0.514966,HEPG2_new_mean_standard_stratified_merge_all_k...,M26
...,...,...,...,...,...,...,...
355,1,0.466667,0.419764,0.453654,0.432074,HEPG2_new_median_standard_stratified_merge_st_...,M63
356,2,0.502222,0.466222,0.477747,0.464903,HEPG2_new_median_standard_stratified_merge_st_...,M63
357,3,0.475556,0.424563,0.449067,0.431817,HEPG2_new_median_standard_stratified_merge_st_...,M63
358,4,0.506667,0.596671,0.507875,0.505327,HEPG2_new_median_standard_stratified_merge_st_...,M63


In [5]:
import pandas as pd
from scipy.stats import friedmanchisquare

# Group by model and collect fold-wise F1s
model_groups = merged_df.groupby("Easy_Model_Name")

# Make a list of lists: each inner list = F1 values from 5 folds of one model
f1_lists = [group["f1"].tolist() for name, group in model_groups]

# Friedman test
stat, p = friedmanchisquare(*f1_lists)

print("Friedman chi-square statistic:", stat)
print("p-value:", p)

if p < 0.05:
    print("Significant differences detected between models (p < 0.05)")
else:
    print("No significant difference detected (p >= 0.05)")


Friedman chi-square statistic: 297.4968036529681
p-value: 1.6665863261716053e-29
Significant differences detected between models (p < 0.05)


In [6]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

# ---- Step 1: compute mean±std for each model ----
agg = merged_df.groupby(["Easy_Model_Name", "Model"]).agg(
    acc_mean=("acc","mean"), acc_std=("acc","std"),
    precision_mean=("precision","mean"), precision_std=("precision","std"),
    recall_mean=("recall","mean"), recall_std=("recall","std"),
    f1_mean=("f1","mean"), f1_std=("f1","std")
).reset_index()

# ---- Step 2: create pivot table for F1 only for statistical tests ----
pivot_df = merged_df.pivot_table(index="fold", columns="Easy_Model_Name", values="f1")
pivot_df = pivot_df.dropna(axis=1, how="any")

# ---- Step 3: identify best model by mean F1 ----
mean_scores = pivot_df.mean(axis=0)
best_model_easy = mean_scores.idxmax()
print(f"Best model (by F1): {best_model_easy}")

# ---- Step 4: Wilcoxon best vs others + Holm correction ----
results = []
for model in pivot_df.columns:
    if model == best_model_easy:
        continue
    stat, p = wilcoxon(pivot_df[best_model_easy], pivot_df[model])
    results.append((model, p))

raw_p = [p for _, p in results]
reject, p_corrected, _, _ = multipletests(raw_p, method='holm')

compare_df = pd.DataFrame({
    "Easy_Model_Name": [m for m,_ in results],
    "Raw_p": raw_p,
    "Holm_corrected_p": p_corrected,
    "Significantly_worse": reject
})

# Mark top group (not significantly worse than best)
top_group = [best_model_easy] + compare_df.loc[compare_df["Significantly_worse"]==False, "Easy_Model_Name"].tolist()

# ---- Step 5: Merge stats table with top group info ----
agg["Top_Group"] = agg["Easy_Model_Name"].isin(top_group)

# Optional: sort by f1_mean
agg = agg.sort_values("f1_mean", ascending=False)

Best model (by F1): M44


In [7]:
# Show summary
print("\n===== Performance Summary (mean ± std) =====")
agg[["Easy_Model_Name","Model",
           "precision_mean","precision_std",
           "recall_mean","recall_std",
           "f1_mean","f1_std",
           "Top_Group"]]


===== Performance Summary (mean ± std) =====


,Easy_Model_Name,Model,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,Top_Group
38,M44,HEPG2_new_median_dmso_mad_structure_merge_all_...,0.626061,0.035623,0.569841,0.016026,0.582713,0.023278,True
12,M20,HEPG2_new_mean_dmso_zscore_structure_merge_all...,0.588299,0.033104,0.570184,0.020492,0.566996,0.020903,True
51,M56,HEPG2_new_median_dmso_zscore_structure_merge_a...,0.637849,0.031042,0.547276,0.037263,0.563943,0.041053,True
70,M8,HEPG2_new_mean_dmso_mad_structure_merge_all_ki...,0.571378,0.077197,0.552579,0.031729,0.552615,0.047304,True
45,M50,HEPG2_new_median_dmso_zscore_stratified_merge_...,0.587119,0.064005,0.540830,0.032918,0.547784,0.030410,True
...,...,...,...,...,...,...,...,...,...
66,M7,HEPG2_new_mean_dmso_mad_structure_as_is,0.350537,0.013686,0.341951,0.016040,0.334769,0.015084,True
63,M67,HEPG2_new_median_standard_structure_as_is,0.331135,0.036823,0.351067,0.033908,0.326665,0.036145,True
43,M49,HEPG2_new_median_dmso_zscore_stratified_as_is,0.312321,0.024117,0.345916,0.030143,0.323180,0.027480,True
4,M13,HEPG2_new_mean_dmso_zscore_stratified_as_is,0.335996,0.029151,0.338201,0.027362,0.322461,0.026165,True


In [8]:
agg.iloc[0:10][[ 'f1_mean', 'f1_std']]


,f1_mean,f1_std
38,0.582713,0.023278
12,0.566996,0.020903
51,0.563943,0.041053
70,0.552615,0.047304
45,0.547784,0.030410
31,0.536444,0.003715
64,0.534607,0.023306
11,0.532945,0.016920
5,0.522173,0.021979
25,0.514172,0.026468


In [9]:
# Condition: rows where Model ends with "_merge_all_kinases"
mask = agg["Model"].str.endswith("_merge_all_kinases")

# Dataframe with only "_merge_all_kinases"
agg_with = agg[mask]

# Dataframe without "_merge_all_kinases"
agg_without = agg[~mask]

In [10]:
agg_without

,Easy_Model_Name,Model,acc_mean,acc_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,Top_Group
32,M39,HEPG2_new_median_dmso_mad_stratified_merge_st_...,0.536889,0.021407,0.534350,0.025646,0.498797,0.044314,0.498665,0.047811,True
52,M57,HEPG2_new_median_dmso_zscore_structure_merge_s...,0.589744,0.016217,0.499609,0.046723,0.502079,0.023320,0.490519,0.026270,True
13,M21,HEPG2_new_mean_dmso_zscore_structure_merge_st_...,0.598974,0.027711,0.504039,0.037948,0.500936,0.016678,0.487220,0.022716,True
39,M45,HEPG2_new_median_dmso_mad_structure_merge_st_v...,0.579487,0.025641,0.500941,0.060115,0.495300,0.043143,0.484869,0.046049,True
22,M3,HEPG2_new_mean_dmso_mad_stratified_merge_st_vs_tk,0.504000,0.015587,0.525957,0.062535,0.481656,0.015284,0.478321,0.023832,True
71,M9,HEPG2_new_mean_dmso_mad_structure_merge_st_vs_tk,0.602051,0.037439,0.471852,0.064905,0.498191,0.048904,0.476229,0.057853,True
6,M15,HEPG2_new_mean_dmso_zscore_stratified_merge_st...,0.503111,0.025415,0.484661,0.054323,0.482408,0.037605,0.474000,0.042347,True
19,M27,HEPG2_new_mean_standard_stratified_merge_st_vs_tk,0.480889,0.031802,0.479591,0.025799,0.473470,0.028545,0.468033,0.026289,True
46,M51,HEPG2_new_median_dmso_zscore_stratified_merge_...,0.495111,0.025647,0.483727,0.034648,0.469059,0.026897,0.464689,0.026256,True
26,M33,HEPG2_new_mean_standard_structure_merge_st_vs_tk,0.546667,0.026548,0.473852,0.046012,0.497503,0.017766,0.460612,0.024177,True


In [11]:
print(agg_without.iloc[0,1])
print(agg_without.iloc[1,1])
print(agg_without.iloc[2,1])
print(agg_without.iloc[3,1])
print(agg_without.iloc[4,1])
print(agg_without.iloc[5,1])
print(agg_without.iloc[6,1])
print(agg_without.iloc[7,1])
print(agg_without.iloc[8,1])
print(agg_without.iloc[9,1])

HEPG2_new_median_dmso_mad_stratified_merge_st_vs_tk
HEPG2_new_median_dmso_zscore_structure_merge_st_vs_tk
HEPG2_new_mean_dmso_zscore_structure_merge_st_vs_tk
HEPG2_new_median_dmso_mad_structure_merge_st_vs_tk
HEPG2_new_mean_dmso_mad_stratified_merge_st_vs_tk
HEPG2_new_mean_dmso_mad_structure_merge_st_vs_tk
HEPG2_new_mean_dmso_zscore_stratified_merge_st_vs_tk
HEPG2_new_mean_standard_stratified_merge_st_vs_tk
HEPG2_new_median_dmso_zscore_stratified_merge_st_vs_tk
HEPG2_new_mean_standard_structure_merge_st_vs_tk


In [ ]:
agg_without.iloc[0:10][[ 'f1_mean', 'f1_std']]
